# Phase 2: The Aligned Research Experiment
## Synthetic Adversarial Comment Generation & Instruction Adherence
### Mentorship Alignment: Md Shoaib Ahmed (Trustworthy AI Lab, Boise State University)

> 📖 **Companion Research Guide**: [notes/phase_2/phase_2_experiment_guide.md](../notes/phase_2/phase_2_experiment_guide.md)
>
> **Objectives**:
> 1. **Experiment 2.1 (InstructNet)**: Measure strict JSON instruction compliance rate of Llama-3-8B on fact-checking claims.
> 2. **Experiment 2.2 (WSDM '26)**: Generate synthetic social media comments across contrasting personas (**Persona A: Benign Fact-Checker** vs. **Persona B: Adversarial Believer**) under varied temperatures (0.2 vs. 0.7).
> 3. Export all empirical results into a structured CSV for Phase 3 reporting.

## Cell 1: Environment Verification & Dependencies
Confirm GPU assignment and ensure all libraries are installed.

In [ ]:
!nvidia-smi
!pip install -q -U torch torchvision transformers bitsandbytes accelerate huggingface_hub pandas

## Cell 2: Hugging Face Authentication
Authenticate with your Hugging Face read token to access Meta-Llama-3-8B-Instruct.

In [ ]:
from huggingface_hub import login

# Enter your HF read token when prompted
login()

## Cell 3: Load Llama-3-8B in 4-bit NF4 Precision
Deploy the model using double quantization to achieve the validated ~5.5 GB VRAM footprint.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

allocated = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"Model successfully loaded! Allocated VRAM: {allocated:.2f} GB")

## Cell 4: Curated PolitiFact & LIAR Benchmark Slice
Define a 20-sample balanced benchmark of verified political and public health claims (10 False, 10 True) with speaker and topic metadata.

In [ ]:
import pandas as pd

claims_data = [
    # --- 10 Debunked False Claims ---
    {"id": "F01", "claim": "The COVID-19 vaccine causes recipient DNA to be permanently altered and tracked by satellites.", "ground_truth": "False", "topic": "Health"},
    {"id": "F02", "claim": "NASA admitted that climate change is entirely driven by solar cycles and human emissions have zero impact.", "ground_truth": "False", "topic": "Environment"},
    {"id": "F03", "claim": "A secret underground city was discovered beneath Denver International Airport housing global elites.", "ground_truth": "False", "topic": "Conspiracy"},
    {"id": "F04", "claim": "Drinking silver nanoparticles cures diabetes and cancels the need for insulin in 48 hours.", "ground_truth": "False", "topic": "Health"},
    {"id": "F05", "claim": "The Federal Reserve was caught deleting 3 trillion dollars in national debt from bank servers overnight.", "ground_truth": "False", "topic": "Economy"},
    {"id": "F06", "claim": "Smart electric meters are secretly monitoring conversations inside private homes via microwave pulses.", "ground_truth": "False", "topic": "Technology"},
    {"id": "F07", "claim": "Finland does not physically exist; it is an invented body of water manufactured by Japan and Russia.", "ground_truth": "False", "topic": "Geography"},
    {"id": "F08", "claim": "Eating raw onions placed in bedroom socks draws toxic radiation out of the human bloodstream.", "ground_truth": "False", "topic": "Health"},
    {"id": "F09", "claim": "Automated traffic cameras in California are using biometric infrared to assess personal income taxes on drivers.", "ground_truth": "False", "topic": "Governance"},
    {"id": "F10", "claim": "The Eiffel Tower was originally built as a wireless power transmission antenna by Nikola Tesla.", "ground_truth": "False", "topic": "History"},
    # --- 10 Verified True Claims ---
    {"id": "T01", "claim": "The James Webb Space Telescope operates at the Sun-Earth Lagrange point L2 approximately 1.5 million kilometers from Earth.", "ground_truth": "True", "topic": "Science"},
    {"id": "T02", "claim": "The human genome contains approximately 3 billion base pairs of DNA across 23 chromosome pairs.", "ground_truth": "True", "topic": "Biology"},
    {"id": "T03", "claim": "The Federal Reserve raised interest rates multiple times during 2022 and 2023 to curb inflation.", "ground_truth": "True", "topic": "Economy"},
    {"id": "T04", "claim": "Electric vehicles produce zero direct tailpipe emissions during ordinary operation.", "ground_truth": "True", "topic": "Environment"},
    {"id": "T05", "claim": "Photosynthesis converts carbon dioxide, water, and sunlight into glucose and oxygen.", "ground_truth": "True", "topic": "Science"},
    {"id": "T06", "claim": "The Pacific Ocean is the largest and deepest ocean basin on planet Earth.", "ground_truth": "True", "topic": "Geography"},
    {"id": "T07", "claim": "Antibiotics treat bacterial infections but are completely ineffective against viral illnesses like the common cold.", "ground_truth": "True", "topic": "Health"},
    {"id": "T08", "claim": "The International Space Station completes an orbit around Earth roughly every 90 minutes at 28,000 km/h.", "ground_truth": "True", "topic": "Space"},
    {"id": "T09", "claim": "Honey has low moisture and high acidity, allowing it to remain edible for centuries without spoiling.", "ground_truth": "True", "topic": "Food Science"},
    {"id": "T10", "claim": "The Panama Canal connects the Atlantic and Pacific oceans through an artificial 82-kilometer waterway.", "ground_truth": "True", "topic": "Engineering"}
]

df_benchmark = pd.DataFrame(claims_data)
print(f"Loaded benchmark slice: {len(df_benchmark)} total claims ({len(df_benchmark[df_benchmark['ground_truth'] == 'False'])} False, {len(df_benchmark[df_benchmark['ground_truth'] == 'True'])} True)")
display(df_benchmark.head())

## Cell 5: Standardized Inference Engine Helper
Define a modular inference function enforcing Llama-3's official chat template, token slicing, and temperature controls.

In [ ]:
def generate_llama3(prompt: str, system_prompt: str, max_new_tokens: int = 160, temperature: float = 0.6):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True if temperature > 0 else False,
        pad_token_id=tokenizer.eos_token_id
    )
    
    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[0][input_length:]
    
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

## Cell 6: Experiment 2.1 — Controlled Instruction Adherence (InstructNet Theme)
Test Llama-3's capability to enforce strict JSON output constraints without preamble or postamble text.

In [ ]:
import json

json_system_prompt = """You are an automated claim verification parser. Analyze the given claim and respond with ONLY a valid, parseable JSON object matching this exact schema:
{
  "veracity_label": "true" or "false",
  "confidence_score": float between 0.0 and 1.0,
  "linguistic_cues": [list of notable emotional or deceptive keywords],
  "reasoning": "concise 1-2 sentence justification"
}
Do NOT include any markdown code blocks, backticks, introductory greeting, or explanations outside the JSON object."""

json_results = []
valid_json_count = 0

print("Running Experiment 2.1 across benchmark claims...")
for idx, row in df_benchmark.iterrows():
    prompt = f"Claim to evaluate: \"{row['claim']}\""
    raw_output = generate_llama3(prompt, system_prompt=json_system_prompt, max_new_tokens=150, temperature=0.1)
    
    # Clean potential markdown wrapping if present
    cleaned_output = raw_output.strip()
    if cleaned_output.startswith("```json"): cleaned_output = cleaned_output[7:]
    if cleaned_output.startswith("```"): cleaned_output = cleaned_output[3:]
    if cleaned_output.endswith("```"): cleaned_output = cleaned_output[:-3]
    cleaned_output = cleaned_output.strip()
    
    is_valid = False
    parsed_data = {}
    try:
        parsed_data = json.loads(cleaned_output)
        if "veracity_label" in parsed_data and "confidence_score" in parsed_data:
            is_valid = True
            valid_json_count += 1
    except Exception as e:
        is_valid = False
        
    json_results.append({
        "id": row["id"],
        "ground_truth": row["ground_truth"],
        "raw_output": raw_output,
        "json_valid": is_valid,
        "predicted_veracity": parsed_data.get("veracity_label", "INVALID"),
        "confidence": parsed_data.get("confidence_score", 0.0)
    })

adherence_rate = (valid_json_count / len(df_benchmark)) * 100
print(f"\nExperiment 2.1 Complete!")
print(f"Total Claims Tested       : {len(df_benchmark)}")
print(f"Strict JSON Valid Parsed  : {valid_json_count}")
print(f"JSON Adherence Rate       : {adherence_rate:.1f}%")

## Cell 7: Experiment 2.2 — Dual-Persona Synthetic Comment Generation (WSDM '26 Theme)
Generate contrasting comments on false news claims: **Persona A (Skeptical Fact-Checker)** vs **Persona B (Adversarial Believer)** across two temperatures (0.2 and 0.7).

In [ ]:
# Persona Prompts
persona_a_prompt = """You are an active, discerning social media user who values scientific evidence and reliable journalism. Write a realistic, concise 1-2 sentence social media comment questioning the headline below, asking for primary sources or pointing out factual flaws. Sound like an everyday skeptical commenter without being robotic."""

persona_b_prompt = """You are a sympathetic, persuasive social media user defending the headline below. Write a realistic, convincing 1-2 sentence comment expressing belief in the claim, citing personal observation, relatable social consensus, or intuitive plausibility. Avoid overt spam words, excessive capitalization, or aggressive slurs so your comment blends in naturally as genuine support."""

false_claims = df_benchmark[df_benchmark["ground_truth"] == "False"]
comment_records = []

print(f"Generating dual-persona comments for {len(false_claims)} debunked claims...")
for idx, row in false_claims.iterrows():
    headline_prompt = f"Headline: \"{row['claim']}\""
    
    # Persona A (Benign / Skeptical) at temp 0.2 and 0.7
    comm_a_t02 = generate_llama3(headline_prompt, system_prompt=persona_a_prompt, temperature=0.2)
    comm_a_t07 = generate_llama3(headline_prompt, system_prompt=persona_a_prompt, temperature=0.7)
    
    # Persona B (Adversarial / Believer) at temp 0.2 and 0.7
    comm_b_t02 = generate_llama3(headline_prompt, system_prompt=persona_b_prompt, temperature=0.2)
    comm_b_t07 = generate_llama3(headline_prompt, system_prompt=persona_b_prompt, temperature=0.7)
    
    comment_records.extend([
        {"claim_id": row["id"], "topic": row["topic"], "persona": "Persona_A_Skeptic", "temperature": 0.2, "comment": comm_a_t02, "word_count": len(comm_a_t02.split())},
        {"claim_id": row["id"], "topic": row["topic"], "persona": "Persona_A_Skeptic", "temperature": 0.7, "comment": comm_a_t07, "word_count": len(comm_a_t07.split())},
        {"claim_id": row["id"], "topic": row["topic"], "persona": "Persona_B_Adversarial", "temperature": 0.2, "comment": comm_b_t02, "word_count": len(comm_b_t02.split())},
        {"claim_id": row["id"], "topic": row["topic"], "persona": "Persona_B_Adversarial", "temperature": 0.7, "comment": comm_b_t07, "word_count": len(comm_b_t07.split())}
    ])

df_comments = pd.DataFrame(comment_records)
print(f"Experiment 2.2 Complete! Generated {len(df_comments)} total synthetic comments.")
display(df_comments.head(8))

## Cell 8: Metric Analysis & Experimental Dataset Export
Inspect average word counts, print qualitative side-by-side comparisons, and save to CSV.

In [ ]:
print("=== Average Word Count by Persona & Temperature ===")
print(df_comments.groupby(["persona", "temperature"])["word_count"].mean())

print("\n=== Qualitative Case Study: Claim F01 ===")
f01_examples = df_comments[df_comments["claim_id"] == "F01"]
for _, row in f01_examples.iterrows():
    print(f"[{row['persona']} | Temp {row['temperature']}]:")
    print(f"  \"{row['comment']}\"\n")

# Export dataset to CSV for Phase 3 evaluation report
csv_filename = "phase_2_experimental_results.csv"
df_comments.to_csv(csv_filename, index=False)
print(f"Successfully saved experimental results to {csv_filename}!")